In [15]:
import pandas as pd

In [17]:
data = pd.read_csv("D:/Công việc/DA + DE/Book DA/Data Science for Business/PaySim/fraud-detection-paysim\data/raw/Synthetic_Financial_datasets_log.csv")
data.head(5)

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\admin\AppData\Local\Temp\ipykernel_1644\3529440422.py:1: SyntaxWarning: invalid escape sequence '\d'
  data = pd.read_csv("D:/Công việc/DA + DE/Book DA/Data Science for Business/PaySim/fraud-detection-paysim\data/raw/Synthetic_Financial_datasets_log.csv")


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [18]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 11 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   step            1048575 non-null  int64  
 1   type            1048575 non-null  object 
 2   amount          1048575 non-null  float64
 3   nameOrig        1048575 non-null  object 
 4   oldbalanceOrg   1048575 non-null  float64
 5   newbalanceOrig  1048575 non-null  float64
 6   nameDest        1048575 non-null  object 
 7   oldbalanceDest  1048575 non-null  float64
 8   newbalanceDest  1048575 non-null  float64
 9   isFraud         1048575 non-null  int64  
 10  isFlaggedFraud  1048575 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 88.0+ MB


In [22]:
data.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1.048575e+06,1048575.0
mean,2.696617e+01,1.586670e+05,8.740095e+05,8.938089e+05,9.781600e+05,1.114198e+06,1.089097e-03,0.0
std,1.562325e+01,2.649409e+05,2.971751e+06,3.008271e+06,2.296780e+06,2.416593e+06,3.298351e-02,0.0
min,1.000000e+00,1.000000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0
25%,1.500000e+01,1.214907e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0
50%,2.000000e+01,7.634333e+04,1.600200e+04,0.000000e+00,1.263772e+05,2.182604e+05,0.000000e+00,0.0
75%,3.900000e+01,2.137619e+05,1.366420e+05,1.746000e+05,9.159235e+05,1.149808e+06,0.000000e+00,0.0
max,9.500000e+01,1.000000e+07,3.890000e+07,3.890000e+07,4.210000e+07,4.220000e+07,1.000000e+00,0.0


Insight 1: Sự lãng phí của kiểu dữ liệu mặc định (isFraud)
Mặc định, Pandas dùng int64 (8 bytes) cho cột isFraud. Nhưng chúng ta biết nó chỉ chứa 0 và 1.

Cách kiểm chứng:
Chúng ta sẽ so sánh dung lượng của cột này trước và sau khi ép về int8 (1 byte).

In [25]:
#1. Tính dung lượng hiện tại (đơn vị: Bytes)
original_size = data['isFraud'].memory_usage(index=False, deep=True)

In [26]:
# 2. Ép kiểu thử nghiệm
downcasted_isFraud = data['isFraud'].astype('int8')
new_size = downcasted_isFraud.memory_usage(index=False, deep=True)

In [27]:
print(f"Dung lượng ban đầu: {original_size/ 1024**2:.2f} MB")
print(f"Dung lượng sau khi tối ưu: {new_size / 1024**2:.2f} MB")
print(f"Tỷ lệ tiết kiệm: {(1- new_size/original_size)*100:.1f}%")

Dung lượng ban đầu: 8.00 MB
Dung lượng sau khi tối ưu: 1.00 MB
Tỷ lệ tiết kiệm: 87.5%


Insight 2: Tối ưu dựa trên dải giá trị thực tế (step)
Cột step đại diện cho số giờ. Trong dữ liệu đầy đủ, nó chỉ tới 743.

Cách kiểm chứng:
Chúng ta kiểm tra giá trị lớn nhất và xem nó có "vừa" với các kiểu dữ liệu nhỏ hơn không.

In [29]:
step_max = data['step'].max()
print(f"Giá trị lớn nhât của step: {step_max} ")

Giá trị lớn nhât của step: 95 


In [31]:
# Kiểm tra giới hạn của int16
import numpy as np
print(f"Giới hạn tối đa của int16: {np.iinfo(np.int16).max}")

Giới hạn tối đa của int16: 32767


Insight 3: Sai số khi dùng float32 cho giao dịch (amount)
Chúng ta sẽ xem liệu việc giảm độ chính xác có làm thay đổi số tiền đáng kể không.

Cách kiểm chứng:
Tính sự chênh lệch (sai số) giữa giá trị 64-bit và 32-bit.

In [32]:
# Tính sai số trung bình 
diff = (data['amount'] - data['amount'].astype('float32')).abs()
print(f"Sai số trung bình trên mỗi giao dịch: {diff.mean():.10f}")

Sai số trung bình trên mỗi giao dịch: 0.0033932988
